In [1]:
import numpy as np
import pandas as pd
import torch

from scipy.special import comb, perm
from itertools import combinations, permutations

from tqdm import tqdm, trange
from pybloom_live import BloomFilter

import math
import sys
sys.path.append("../..") ## 定位到utils目录

from torch.nn.utils.rnn import pad_sequence
from torch.nn.functional import one_hot
import torch
import networkx as nx

from utils import utils, SubGDataset

### GM12878 Unobserved 3way MCI. (所有bin的3way组合 - Observed 3way MCI)


In [6]:
data_dir = '/data/xujs/Project/DeepLearning/MCIP/Results'
chrom_range_ = np.load(f"{data_dir}/preprocess_results/hg38.1000kb.chrom_range.npy")
chrom_range = {}
for x in chrom_range_:
    chrom_range[x[0]] = np.array([int(x[1])-1, int(x[2])-1])

num = []
for chrom, v in chrom_range.items():
    num.append(v[1] - v[0])

node2chrom = np.load(f"{data_dir}/preprocess_results/hg38.1000kb.node2chrom.npy", allow_pickle=True).item()
node2chrom = {(node-1):chrom for node, chrom in node2chrom.items()}

node_num = len(node2chrom)

In [419]:
## HiPore-C GM12878 1Mb order:3.

chroms = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10',
            'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 
            'chr21', 'chr22', 'chrX']

order_list = [3, 4, 5]
order = 3
min_dis = 5

unobserved_samples = []

for chrom in tqdm(chroms):
    node_start, node_end = chrom_range[chrom][0], chrom_range[chrom][1]
    chrom_unobserved_samples = np.array(list(combinations(np.arange(node_start, node_end), 3)))
    node_dis = np.diff(chrom_unobserved_samples)
    node_min_dis = np.min(node_dis, axis=1)
    chrom_unobserved_samples = chrom_unobserved_samples[node_min_dis >= min_dis]
    chrom_unobserved_samples = set([tuple(x) for x in chrom_unobserved_samples])


    data = np.load(f'/data/xujs/Project/DeepLearning/MCIP/Results/HiPore-C_GM12878/1000kb/chrom_subgraph/{chrom}_subgraph.npy', allow_pickle=True)
    # data_weight = np.load(f'/data/xujs/Project/DeepLearning/MCIP/Results/HiPore-C_GM12878/1000kb/chrom_subgraph/{chrom}_subgraph_freq.npy', allow_pickle=True)
    # data_order = np.array([len(x) for x in data])
    decomposed_data = []

    for datum in data:
        decomposed_ = list(combinations(datum, order))
        decomposed_data += decomposed_

    decomposed_data = set(decomposed_data)

    chrom_unobserved_samples = list(chrom_unobserved_samples.difference(decomposed_data))
    unobserved_samples.append(chrom_unobserved_samples)

unobserved_samples = np.concatenate(unobserved_samples)
np.save('./HiPore-C_GM12878_1Mb.Unobserved_MCI_O3.npy', unobserved_samples)

100%|██████████| 23/23 [01:09<00:00,  3.02s/it]


In [7]:
import numpy as np
import pandas as pd
from torch.nn.utils.rnn import pad_sequence
from torch.nn.functional import one_hot
import torch
from torch_geometric.utils import is_undirected, to_undirected, negative_sampling, to_networkx
from torch_geometric.data import Data
import networkx as nx
import os

class BaseGraph(Data):
    def __init__(self, x, edge_index, edge_weight, subG_node, subG_label, subG_weight, mask):
        '''
        A general format for datasets.
        Args:
            x: node feature. For our used datasets, x is empty vector.
            subG_node: a matrix like [[0,2,3],[1,4,5],[6,7,-1]], whose i-th row contains the nodes in the i-th subgraph. -1 is for padding.
            subG_label: the target of subgraphs.
            mask: of shape (number of subgraphs), type torch.long. mask[i]=0,1,2 if i-th subgraph is in the training set, validation set and test set respectively. 
        '''
        super(BaseGraph, self).__init__(x=x,
                                        edge_index=edge_index,
                                        edge_attr=edge_weight,
                                        pos=subG_node,
                                        y=subG_label)
        self.subG_weight = subG_weight
        self.mask = mask
        self.to_undirected()

    def setDegreeFeature(self, mod=1):
        # use node degree as node features.
        adj = torch.sparse_coo_tensor(self.edge_index, self.edge_attr,
                                        (self.x.shape[0], self.x.shape[0]))
        degree = torch.sparse.sum(adj, dim=1).to_dense().to(torch.int64)
        degree = torch.div(degree, mod, rounding_mode='floor')
        degree = torch.unique(degree, return_inverse=True)[1]
        self.x = degree.reshape(self.x.shape[0], 1, -1)

    def setOneFeature(self):
        # use homogeneous node features.
        self.x = torch.ones((self.x.shape[0], 1, 1), dtype=torch.int64)

    def setNodeIdFeature(self):
        # use nodeid as node features.
        self.x = torch.arange(self.x.shape[0], dtype=torch.int64).reshape(
            self.x.shape[0], 1, -1)

    def get_split(self, split: str):
        tar_mask = {"train": 0, "valid": 1, "test": 2}[split]
        return self.x, self.edge_index, self.edge_attr, self.pos[
            self.mask == tar_mask], self.y[self.mask == tar_mask], self.subG_weight[self.mask == tar_mask]

    def get_data(self):
        return self.x, self.edge_index, self.edge_attr, self.pos, self.y, self.subG_weight

    def to_undirected(self):
        if not is_undirected(self.edge_index):
            self.edge_index, self.edge_attr = to_undirected(
                self.edge_index, self.edge_attr)

    def get_LPdataset(self, use_loop=False):
        # generate link prediction dataset for pretraining GNNs
        neg_edge = negative_sampling(self.edge_index)
        x = self.x
        ei = self.edge_index
        ea = self.edge_attr
        pos = torch.cat((self.edge_index, neg_edge), dim=1).t()
        y = torch.cat((torch.ones(ei.shape[1]),
                        torch.zeros(neg_edge.shape[1]))).to(ei.device)

        return x, ei, ea, pos, y

    def to(self, device):
        self.x = self.x.to(device)
        self.edge_index = self.edge_index.to(device)
        self.edge_attr = self.edge_attr.to(device)
        self.pos = self.pos.to(device)
        self.y = self.y.to(device)
        self.mask = self.mask.to(device)
        self.subG_weight = self.subG_weight.to(device)
        return self


def tfunc(ds, bs, shuffle=True, drop_last=True):
    return SubGDataset.ZGDataloader(ds,
                                    bs,
                                    z_fn=utils.MaxZOZ,
                                    shuffle=shuffle,
                                    drop_last=drop_last)

def loader_fn(ds, bs):
    return tfunc(ds, bs)

def tloader_fn(ds, bs):
    return tfunc(ds, bs, True, False)

In [1]:
### Unobserbed O3 MCI
name = 'HiPore-C_GM12878_1Mb'

sub_G = np.load(f'./{name}.Unobserved_MCI_O3.npy')
sub_G_label = torch.zeros(sub_G.shape[0], dtype=torch.int64)
sub_G_weight = torch.ones(sub_G.shape[0], dtype=torch.int64)
mask = torch.zeros(sub_G.shape[0], dtype=torch.int64)
pos = pad_sequence([torch.tensor(i) for i in sub_G], batch_first=True, padding_value=-1)

rawedge = nx.read_edgelist(f"/data/xujs/Project/HiC2PoreC/code/SGMCI/dataset/{name}/edge_list.txt", nodetype=int, data=(('weight',float),)).edges(data=True)
edge_index = torch.tensor([[int(i[0]), int(i[1])] for i in rawedge]).t()
edge_weight = torch.tensor([i[2]['weight'] for i in rawedge])


## 需要指定节点数量
genome = 'hg38'
binsize = '1Mb'
batch_size = 512
name = 'HiPore-C_GM12878_1Mb'
df_node_num = pd.read_table(f'/data/xujs/Project/HiC2PoreC/code/SGMCI/dataset/{genome}.{binsize}.node_num.txt')
node_num_dict = dict(zip(df_node_num['chrom'], df_node_num['chrom_node_num']))
chrom_symbol = name.split('_')[-2]
chrom_node_num = node_num_dict[chrom_symbol] if chrom_symbol in node_num_dict else sum(list(node_num_dict.values())) ##低分辨率下不分染色体，而是全基因组

num_node = max([torch.max(pos), torch.max(edge_index)]) + 1
num_node = max([num_node, chrom_node_num])
x = torch.empty((num_node, 1, 0))

## baseG
device = torch.device(f'cuda:{0}' if torch.cuda.is_available() else 'cpu')
baseG = BaseGraph(x, edge_index, edge_weight, pos, sub_G_label, sub_G_weight, mask)
baseG.setNodeIdFeature()
baseG.to(device)
baseG_dataset = SubGDataset.GDataset(*baseG.get_data())
baseG_loader = tloader_fn(baseG_dataset, batch_size)

## de novo predict
model_dir = '/data/xujs/Project/HiC2PoreC/code/SGMCI/results/HiPore-C_GM12878_1Mb/HiPore-C_GM12878_1Mb_test_chr1'
model = torch.load(f'{model_dir}/HiPore-C_GM12878_1Mb_test_chr1_ND_MIX_Struc_r0.pt')
pred_list = []
model.eval()
with torch.no_grad():
    for batch in tqdm(baseG_loader):
        pred, _ = model(*batch[:-2])
        pred_list.append(pred)

preds = torch.cat(pred_list, dim=0)
preds = preds.cpu().detach().numpy()

np.save('./HiPore-C_GM12878_1Mb.Unobserved_MCI_O3_pred_score.npy', preds)

In [7]:
preds = np.load('HiPore-C_GM12878_1Mb.Unobserved_MCI_O3_pred_score.npy')

In [8]:
def filter_by_dis(data, min_dis=5):
    node_dis = np.diff(data)
    node_min_dis = np.min(node_dis, axis=1)
    filtered_data = data[node_min_dis >= min_dis]
    return filtered_data

In [10]:
#### 预测的3way-MCI是否有Hi-C支持
#       1). occ_freq < cutoff
#       2). unobserved O3_MCI

##################### 1) occ_freq < cutoff #####################
### 加载数据 order=3
data_dir = '/data/xujs/Project/DeepLearning/MCIP/Results/'
order, cutoff = 3, 1
data = np.load(f'{data_dir}/HiPore-C_GM12878/1000kb/all_{order}_NotDecompose_subgraph.npy', allow_pickle=True).astype('int')
data = data - 1
data_weight = np.load(f'{data_dir}/HiPore-C_GM12878/1000kb/all_{order}_NotDecompose_subgraph_freq.npy', allow_pickle=True).astype('float32')

data = data[data_weight > cutoff]

### 数据处理 + 模型预测
name = 'HiPore-C_GM12878_1Mb'
# sub_G = np.load(f'./{name}.Unobserved_MCI_O3.npy')
sub_G = filter_by_dis(data, min_dis=5)
sub_G_label = torch.zeros(sub_G.shape[0], dtype=torch.int64)
sub_G_weight = torch.ones(sub_G.shape[0], dtype=torch.int64)
mask = torch.zeros(sub_G.shape[0], dtype=torch.int64)
pos = pad_sequence([torch.tensor(i) for i in sub_G], batch_first=True, padding_value=-1)

rawedge = nx.read_edgelist(f"/data/xujs/Project/HiC2PoreC/code/SGMCI/dataset/{name}/edge_list.txt", nodetype=int, data=(('weight',float),)).edges(data=True)
edge_index = torch.tensor([[int(i[0]), int(i[1])] for i in rawedge]).t()
edge_weight = torch.tensor([i[2]['weight'] for i in rawedge])


## 需要指定节点数量
genome = 'hg38'
binsize = '1Mb'
batch_size = 512
df_node_num = pd.read_table(f'/data/xujs/Project/HiC2PoreC/code/SGMCI/dataset/{genome}.{binsize}.node_num.txt')
node_num_dict = dict(zip(df_node_num['chrom'], df_node_num['chrom_node_num']))
chrom_symbol = name.split('_')[-2]
chrom_node_num = node_num_dict[chrom_symbol] if chrom_symbol in node_num_dict else sum(list(node_num_dict.values())) ##低分辨率下不分染色体，而是全基因组

num_node = max([torch.max(pos), torch.max(edge_index)]) + 1
num_node = max([num_node, chrom_node_num])
x = torch.empty((num_node, 1, 0))

## baseG
device = torch.device(f'cuda:{0}' if torch.cuda.is_available() else 'cpu')
baseG = BaseGraph(x, edge_index, edge_weight, pos, sub_G_label, sub_G_weight, mask)
baseG.setNodeIdFeature()
baseG.to(device)
baseG_dataset = SubGDataset.GDataset(*baseG.get_data())
baseG_loader = tloader_fn(baseG_dataset, batch_size)

## de novo predict
pred_list = []
model.eval()
with torch.no_grad():
    for batch in tqdm(baseG_loader):
        pred, _ = model(*batch[:-2])
        pred_list.append(pred)

preds = torch.cat(pred_list, dim=0)
preds = preds.cpu().detach().numpy()

# np.save('./HiPore-C_GM12878_1Mb.MCI_O3_freq_lt_cutoff.npy', sub_G)
# np.save('./HiPore-C_GM12878_1Mb.MCI_O3_freq_lt_cutoff.pred_score.npy', preds)

100%|██████████| 835/835 [01:05<00:00, 12.71it/s]


In [9]:
#### HiPore-C GM12878 1Mb Unobserved_MCI_O3: SGMCI vs SGMCI

Unobserved_MCI_O3 = np.load('/data/xujs/Project/HiC2PoreC/code/SGMCI/Analysis_Results/2024-05-23_Predict_New_MCI/HiPore-C_GM12878_1Mb.Unobserved_MCI_O3.npy')
SGMCI_O3_preds_score = np.load('/data/xujs/Project/HiC2PoreC/code/SGMCI/Analysis_Results/2024-05-23_Predict_New_MCI/HiPore-C_GM12878_1Mb.Unobserved_MCI_O3_pred_score.npy')
SGMCI_O3_preds_score = SGMCI_O3_preds_score.reshape(-1)

In [12]:
MCI_O3_freq_lt_cutoff = np.load('./HiPore-C_GM12878_1Mb.MCI_O3_freq_lt_cutoff.npy')
MCI_O3_freq_lt_cutoff_preds_score = np.load('./HiPore-C_GM12878_1Mb.MCI_O3_freq_lt_cutoff.pred_score.npy')
MCI_O3_freq_lt_cutoff_preds_score = MCI_O3_freq_lt_cutoff_preds_score.reshape(-1)

In [10]:
### 参考SGMCI: Hi-C edge在1Mb下的定义
'''
We first decomposed all the triplets in the training data (triplets with occurrence frequency >=8) into pairwise edges and
identified the corresponding entry in the Hi-C contact matrix. We then calculated the average value of these entries for 
intra-chromosomal interactions and inter-chromosomal interactions, respectively. 
These two averaged values were then used to binarize the Hi-C contact matrix and build the Hi-C graph.
'''

import cooler

cool_file = '/data/xujs/Project/HiPoreC_Promoter_Result/Result/2022-09-27_HiPoreC_to_cool_hic_files/GM12878.Merge.HiPoreC.mcool::resolutions/1000000'
c = cooler.Cooler(cool_file)

### 得到所有交互
wg_mat = np.zeros((node_num, node_num)) # whole genome mat.
for chrom1, range1 in chrom_range.items():
    # s1, e1 = range1 - 1
    s1, e1 = range1
    for chrom2, range2 in chrom_range.items():
        # s2, e2 = range2 - 1
        s2, e2 = range2 
        mat = c.matrix(balance=False, sparse=False).fetch(chrom1, chrom2)
        wg_mat[s1:e1, s2:e2] = mat

### 得到染色体内交互
intra_adj_dict = {}
for chrom in chrom_range.keys():
    mat = c.matrix(balance=False, sparse=False).fetch(chrom, chrom)
    intra_adj_dict[chrom] = mat

### 得到染色体间交互
inter_adj_dict = {}
for chrom1 in chrom_range.keys():
    for chrom2 in chrom_range.keys():
        if chrom1 != chrom2:
            mat = c.matrix(balance=False, sparse=False).fetch(chrom1, chrom2)
            inter_adj_dict[f'{chrom1}_{chrom2}'] = mat

intra_mean_value_dict = {chrom: np.nanmedian(adj) for chrom, adj in intra_adj_dict.items()}
inter_mean_value_dict = {chrom: np.nanmedian(adj) for chrom, adj in inter_adj_dict.items()}
intra_mean_value = np.mean(list(intra_mean_value_dict.values()))
inter_mean_value = np.mean(list(inter_mean_value_dict.values()))

In [11]:
## 统计支持MCI的Hi-C edges.

def stat_hic_edge(data):

    hic_edge_num_list = []

    for mci in data:
        decomposed_edges = list(combinations(mci, 2))
        n = 0
        for edge in decomposed_edges:
            chrom1, chrom2 = node2chrom[edge[0]], node2chrom[edge[1]]
            hic_contact_freq = wg_mat[edge[0], edge[1]]
            if chrom1 == chrom2:
                if hic_contact_freq >= intra_mean_value_dict[chrom]:
                # if hic_contact_freq >= intra_mean_value:
                    n += 1
            else:
                if hic_contact_freq >= inter_mean_value_dict[f'{chrom1}_{chrom2}']:
                # if hic_contact_freq >= inter_mean_value:
                    n += 1
        hic_edge_num_list.append(n)

    hic_edge_num_list = np.array(hic_edge_num_list)

    edge_num, count = np.unique(hic_edge_num_list, return_counts=True)
    perc = count / count.sum()
    
    df = pd.DataFrame({
        'edge_num': edge_num,
        'count': count,
        'perc': perc
    })

    return df

In [12]:
data_3 = np.load('/data/xujs/Project/DeepLearning/MCIP/Results/HiPore-C_GM12878/1000kb/all_3_NotDecompose_subgraph.npy')
data_3 = data_3 - 1
data_weight_3 = np.load('/data/xujs/Project/DeepLearning/MCIP/Results/HiPore-C_GM12878/1000kb/all_3_NotDecompose_subgraph_freq.npy')

pos = data_3[(data_weight_3 >= 1) & ((data_weight_3 < 8))]
pos_hic_edge = stat_hic_edge(pos)

In [17]:
SGMCI_O3_preds_score_09_10_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.9) & (SGMCI_O3_preds_score < 1.0)])
SGMCI_O3_preds_score_08_09_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.8) & (SGMCI_O3_preds_score < 0.9)])
SGMCI_O3_preds_score_07_08_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.7) & (SGMCI_O3_preds_score < 0.8)])
SGMCI_O3_preds_score_06_07_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.6) & (SGMCI_O3_preds_score < 0.7)])
SGMCI_O3_preds_score_05_06_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.5) & (SGMCI_O3_preds_score < 0.6)])
SGMCI_O3_preds_score_04_05_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.4) & (SGMCI_O3_preds_score < 0.5)])
SGMCI_O3_preds_score_03_04_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.3) & (SGMCI_O3_preds_score < 0.4)])
SGMCI_O3_preds_score_02_03_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.2) & (SGMCI_O3_preds_score < 0.3)])
SGMCI_O3_preds_score_01_02_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.1) & (SGMCI_O3_preds_score < 0.2)])
SGMCI_O3_preds_score_00_01_hic_edge = stat_hic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.0) & (SGMCI_O3_preds_score < 0.1)])

In [59]:
df_SGMCI_O3_hic_edge_group = pd.concat([
                        SGMCI_O3_preds_score_00_01_hic_edge,
                        SGMCI_O3_preds_score_01_02_hic_edge,
                        SGMCI_O3_preds_score_02_03_hic_edge,
                        SGMCI_O3_preds_score_03_04_hic_edge,
                        SGMCI_O3_preds_score_04_05_hic_edge,
                        SGMCI_O3_preds_score_05_06_hic_edge,
                        SGMCI_O3_preds_score_06_07_hic_edge,
                        SGMCI_O3_preds_score_07_08_hic_edge,
                        SGMCI_O3_preds_score_08_09_hic_edge,
                        SGMCI_O3_preds_score_09_10_hic_edge,
                        pos_hic_edge
                    ])
df_SGMCI_O3_hic_edge_group['prob_group'] = ['00_01']*4 + ['01_02']*4 + ['02_03']*4 + ['03_04']*4 + ['04_05']*4 + \
                                            ['05_06']*4 + ['06_07']*4 + ['07_08']*4 + ['08_09']*4 + ['09_10']*4 + ['Pos'] *4
df_SGMCI_O3_hic_edge_group.to_csv('./HiPore-C_GM12878_1Mb.SGMCI_Unobserved_MCI_O3_Hi-C_edge_prob_group.txt', sep='\t', index=False)

In [13]:
## 统计支持MCI的scHi-C edges.
import glob
scool_data_dir = '/data/xujs/Project/HiC2PoreC/code/SGMCI/Analysis_Results/2024-05-20_scHi-C/hg19_scool/hg19_to_hg38_scool'
GM12878_scHiC_files = glob.glob(f'{scool_data_dir}/*GM12878*mcool')

# cool = cooler.Cooler(f'{scool_data_dir}/ramani_10kb_human_8973_GM12878_AAGCGACC-GCCATTAA_10000.matrix.hg19_to_hg38.mcool::resolutions/1000000')

def extract_hic_wg_mat(mcool_file):
    cool = cooler.Cooler(f'{mcool_file}::resolutions/1000000')
    wg_mat = np.zeros((node_num, node_num)) # whole genome mat.

    for chrom1, range1 in chrom_range.items():
        s1, e1 = range1
        for chrom2, range2 in chrom_range.items():
            s2, e2 = range2
            mat = cool.matrix(balance=False, sparse=False).fetch(chrom1, chrom2)
            wg_mat[s1:e1, s2:e2] = mat
        
    return wg_mat


scHiC_matrices = []

for scHiC_file in tqdm(GM12878_scHiC_files):
    wg_mat = extract_hic_wg_mat(mcool_file=scHiC_file)
    scHiC_matrices.append(wg_mat)

100%|██████████| 582/582 [19:50<00:00,  2.04s/it]


In [15]:
## 统计支持MCI的scHi-C edges.

def stat_schic_edge(data):

    schic_edge_list = []

    for mci in data:
        decomposed_edges = list(combinations(mci, 2))
        n = 0
        for edge in decomposed_edges:
            for schic_mat in scHiC_matrices:
                contact_freq = schic_mat[edge[0], edge[1]]
                if contact_freq:
                    n += 1
                    break
                else:
                    continue
        if n == 3:
            schic_edge_list.append(1)
        else:
            schic_edge_list.append(0)

    schic_edge_list = np.array(schic_edge_list)

    schic_support, count = np.unique(schic_edge_list, return_counts=True)
    perc = count / count.sum()
    
    df = pd.DataFrame({
        'schic_support': schic_support,
        'count': count,
        'perc': perc
    })

    return df

In [62]:
SGMCI_O3_preds_score_00_01_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.0) & (SGMCI_O3_preds_score < 0.1)])
SGMCI_O3_preds_score_01_02_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.1) & (SGMCI_O3_preds_score < 0.2)])
SGMCI_O3_preds_score_02_03_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.2) & (SGMCI_O3_preds_score < 0.3)])
SGMCI_O3_preds_score_03_04_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.3) & (SGMCI_O3_preds_score < 0.4)])
SGMCI_O3_preds_score_04_05_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.4) & (SGMCI_O3_preds_score < 0.5)])
SGMCI_O3_preds_score_05_06_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.5) & (SGMCI_O3_preds_score < 0.6)])
SGMCI_O3_preds_score_06_07_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.6) & (SGMCI_O3_preds_score < 0.7)])
SGMCI_O3_preds_score_07_08_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.7) & (SGMCI_O3_preds_score < 0.8)])
SGMCI_O3_preds_score_08_09_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.8) & (SGMCI_O3_preds_score < 0.9)])
SGMCI_O3_preds_score_09_10_schic_edge = stat_schic_edge(Unobserved_MCI_O3[(SGMCI_O3_preds_score >= 0.9) & (SGMCI_O3_preds_score < 1.0)])


In [24]:
pos = data_3[(data_weight_3 >= 2)]
index = np.arange(pos.shape[0])
pos = pos[np.random.choice(index, size=20000)]
pos_schic_edge = stat_schic_edge(pos)

In [25]:
pos_schic_edge

,schic_support,count,perc
0,0,11114,0.5557
1,1,8886,0.4443


In [2]:

df_SGMCI_O3_schic_edge_group = pd.concat([ SGMCI_O3_preds_score_00_01_schic_edge,
                                            SGMCI_O3_preds_score_01_02_schic_edge,
                                            SGMCI_O3_preds_score_02_03_schic_edge,
                                            SGMCI_O3_preds_score_03_04_schic_edge,
                                            SGMCI_O3_preds_score_04_05_schic_edge,
                                            SGMCI_O3_preds_score_05_06_schic_edge,
                                            SGMCI_O3_preds_score_06_07_schic_edge,
                                            SGMCI_O3_preds_score_07_08_schic_edge,
                                            SGMCI_O3_preds_score_08_09_schic_edge,
                                            SGMCI_O3_preds_score_09_10_schic_edge,
                                            pos_schic_edge])
df_SGMCI_O3_schic_edge_group['prob_group'] =   ['00_01']*2 + ['01_02']*2 + ['02_03']*2 + ['03_04']*2 + ['04_05']*2 + \
                                                ['05_06']*2 + ['06_07']*2 + ['07_08']*2 + ['08_09']*2 + ['09_10']*2 + ['Pos']*2
df_SGMCI_O3_schic_edge_group.to_csv('./HiPore-C_GM12878_1Mb.SGMCI_Unobserved_MCI_O3_scHi-C_edge_prob_group.txt', sep='\t', index=False)

In [84]:
df_SGMCI_O3_schic_edge_group

,schic_support,count,perc,prob_group
0,0,4297877,0.960832,00_01
1,1,175203,0.039168,00_01
0,0,214142,0.921358,01_02
1,1,18278,0.078642,01_02
0,0,99799,0.912607,02_03
1,1,9557,0.087393,02_03
0,0,61199,0.905672,03_04
1,1,6374,0.094328,03_04
0,0,41573,0.899420,04_05
1,1,4649,0.100580,04_05
